In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import keras
import matplotlib.pyplot as plt
import pandas as pd

from keras.models import Model
from keras.layers import (
    Input,
    Dense,
    Flatten,
    Conv2D,
    MaxPooling2D,
    Dropout,
    BatchNormalization,
    GlobalAveragePooling2D,
    Activation
)
from keras.optimizers import Adam
from keras.utils import plot_model
from keras.datasets import cifar10
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# ============================================================
# CLASS LABELS
# ============================================================

labels = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
]

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

(x_train_full, y_train_full), (x_test, y_test) = cifar10.load_data()

print("Training data shape:", x_train_full.shape)
print("Test data shape:", x_test.shape)

In [ ]:
# ============================================================
# DISPLAY SAMPLE IMAGE
# ============================================================

plt.figure(figsize=(4,4))
plt.imshow(x_train_full[0])
plt.title(f"Label: {labels[y_train_full[0][0]]}")
plt.axis("off")
plt.show()


In [ ]:
# ============================================================
# DATA PREPROCESSING
# ============================================================

# Convert to float32
x_train_full = x_train_full.astype('float32')
x_test = x_test.astype('float32')

# Normalize to range [0,1]
x_train_full = x_train_full / 255.0
x_test = x_test / 255.0

# ============================================================
# SPLIT DATA
# 60% TRAIN
# 20% VALIDATION
# 20% TEST
# ============================================================

# Split all train data into train and val datasets
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full,
    y_train_full,
    test_size=0.20,
    random_state=42
)

print("Train shape:", x_train.shape)
print("Validation shape:", x_val.shape)
print("Test shape:", x_test.shape)

In [ ]:
# ============================================================
# ONE HOT ENCODING - change to binary
# ============================================================

num_classes = 10

y_train_cat = keras.utils.to_categorical(y_train, num_classes)
y_val_cat = keras.utils.to_categorical(y_val, num_classes)
y_test_cat = keras.utils.to_categorical(y_test, num_classes)


In [ ]:
# ============================================================
# DATA AUGMENTATION
# ============================================================

data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.1)
])


#MODEL 1

In [ ]:
# ============================================================
# MODEL 1
# ============================================================

input_shape = (32, 32, 3)

inputs1 = Input(shape=input_shape)

x1 = data_augmentation(inputs1)

x1 = Conv2D(32, (3,3), padding='same', activation='relu')(x1)
x1 = MaxPooling2D((2,2))(x1)

x1 = Conv2D(64, (3,3), padding='same', activation='relu')(x1)
x1 = MaxPooling2D((2,2))(x1)

x1 = Flatten()(x1)

x1 = Dense(128, activation='relu')(x1)
x1 = Dropout(0.5)(x1)

outputs1 = Dense(num_classes, activation='softmax')(x1)

model1 = Model(inputs=inputs1, outputs=outputs1)

In [ ]:
model1.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=1e-2),
    metrics=['accuracy']
)

#MODEL 2

In [ ]:
# ============================================================
# MODEL 2 - more layers
# ============================================================

input_shape = (32, 32, 3)

inputs2 = Input(shape=input_shape)

x2 = data_augmentation(inputs2)

x2 = Conv2D(32, (3,3), padding='same')(x2)
x2 = BatchNormalization()(x2)
x2 = Activation('relu')(x2)
x2 = MaxPooling2D((2,2))(x2)
x2 = Dropout(0.25)(x2)

x2 = Conv2D(64, (3,3), padding='same')(x2)
x2 = BatchNormalization()(x2)
x2 = Activation('relu')(x2)
x2 = MaxPooling2D((2,2))(x2)
x2 = Dropout(0.30)(x2)

x2 = Conv2D(128, (3,3), padding='same')(x2)
x2 = BatchNormalization()(x2)
x2 = Activation('relu')(x2)
x2 = MaxPooling2D((2,2))(x2)
x2 = Dropout(0.4)(x2)

x2 = Flatten()(x2)

x2 = Dense(128)(x2)
x2 = BatchNormalization()(x2)
x2 = Activation('relu')(x2)
x2 = Dropout(0.5)(x2)

outputs2 = Dense(num_classes, activation='softmax')(x2)

model2 = Model(inputs=inputs2, outputs=outputs2)

In [ ]:
model2.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=1e-2),
    metrics=['accuracy']
)

#TRAIN MODELS

MODEL 1

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True
)

rlp = keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.5,
    patience=9,
    min_delta=1e-3,
    min_lr=1e-4,
    mode='min',
)

history1 = model1.fit(
    x_train,
    y_train_cat,
    batch_size=128,
    epochs=60,
    validation_data=(x_val, y_val_cat),
    callbacks=[early_stop,rlp],
    verbose=1
)

MODEL 2

In [ ]:
history2 = model2.fit(
    x_train,
    y_train_cat,
    batch_size=128,
    epochs=60,
    validation_data=(x_val, y_val_cat),
    callbacks=[early_stop,rlp],
    verbose=1
)

#TRAIN HISTORY

In [ ]:
# ============================================================
# PLOT TRAINING HISTORY
# ============================================================

# MODEL 1
df1 = pd.DataFrame(history1.history)

plt.figure(figsize=(10,5))
plt.plot(df1['loss'], label='Train Loss')
plt.plot(df1['val_loss'], label='Validation Loss')
plt.title('MODEL 1 - LOSS')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.show()

# MODEL 2
df2 = pd.DataFrame(history2.history)

plt.figure(figsize=(10,5))
plt.plot(df2['loss'], label='Train Loss')
plt.plot(df2['val_loss'], label='Validation Loss')
plt.title('MODEL 2 - LOSS')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# ============================================================
# MODEL EVALUATION
# ============================================================

loss1, accuracy1 = model1.evaluate(x_test, y_test_cat, verbose='auto')

print("\nMODEL 1 RESULTS")
print("Test loss:", loss1)
print("Test accuracy:", accuracy1)

loss2, accuracy2 = model2.evaluate(x_test, y_test_cat, verbose='auto')

print("\nMODEL 2 RESULTS")
print("Test loss:", loss2)
print("Test accuracy:", accuracy2)

In [ ]:
# ============================================================
# PREDICTIONS
# ============================================================

predictions1 = np.argmax(model1.predict(x_test), axis=1)
predictions2 = np.argmax(model2.predict(x_test), axis=1)

y_test_flat = y_test.flatten()

In [ ]:
# ============================================================
# VISUALIZATION OF TRAINING PROCESS
# ============================================================
# MODEL 1

fig_loss_acc = plt.figure(figsize=(10,4))

# LOSS
plt.subplot(1,2,1)

plt.plot(history1.history['loss'], label='loss (train)')
plt.plot(history1.history['val_loss'], label='loss (validation)')

plt.xlabel('Epoch')
plt.ylabel('Categorical Crossentropy')
plt.title(f'Model 1 loss\nTest loss = {loss1:.4f}')

plt.legend()

# ACCURACY
plt.subplot(1,2,2)

plt.plot(history1.history['accuracy'], label='accuracy (train)')
plt.plot(history1.history['val_accuracy'], label='accuracy (validation)')

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title(f'Model 1 accuracy\nTest accuracy = {accuracy1:.4f}')

plt.legend()

plt.tight_layout()
plt.show()

# MODEL 2
fig_loss_acc = plt.figure(figsize=(10,4))

# LOSS
plt.subplot(1,2,1)

plt.plot(history2.history['loss'], label='loss (train)')
plt.plot(history2.history['val_loss'], label='loss (validation)')

plt.xlabel('Epoch')
plt.ylabel('Categorical Crossentropy')
plt.title(f'Model 2 loss\nTest loss = {loss2:.4f}')

plt.legend()

# ACCURACY
plt.subplot(1,2,2)

plt.plot(history2.history['accuracy'], label='accuracy (train)')
plt.plot(history2.history['val_accuracy'], label='accuracy (validation)')

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title(f'Model 2 accuracy\nTest accuracy = {accuracy2:.4f}')

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# MISCLASSIFIED EXAMPLES
# ============================================================
# MODEL 1
# Predictions
pred_probs = model1.predict(x_test)

# Convert probabilities -> labels
predictions1 = np.argmax(pred_probs, axis=1)

# True labels
y_test_flat = y_test.flatten()

# Incorrect classifications
incorrect_indices = np.nonzero(predictions1 != y_test_flat)[0]

# Show examples
n_show = min(5, len(incorrect_indices))

for i in range(n_show):

    idx = incorrect_indices[i]

    plt.figure(figsize=(4,4))

    plt.imshow(x_test[idx])

    plt.title(
        f"Model 1\n"
        f"True: {labels[y_test_flat[idx]]}\n"
        f"Predicted: {labels[predictions1[idx]]}"
    )

    plt.axis('off')
    plt.show()

# MODEL 2
pred_probs = model2.predict(x_test)

# Convert probabilities -> labels
predictions2 = np.argmax(pred_probs, axis=1)

# True labels
y_test_flat = y_test.flatten()

# Incorrect classifications
incorrect_indices = np.nonzero(predictions2 != y_test_flat)[0]

# Show examples
n_show = min(5, len(incorrect_indices))

for i in range(n_show):

    idx = incorrect_indices[i]

    plt.figure(figsize=(4,4))

    plt.imshow(x_test[idx])

    plt.title(
        f"Model 2\n"
        f"True: {labels[y_test_flat[idx]]}\n"
        f"Predicted: {labels[predictions2[idx]]}"
    )

    plt.axis('off')
    plt.show()

In [ ]:
# ============================================================
# CONFUSION MATRIX
# ============================================================
# MODEL 1
cm1 = confusion_matrix(y_test_flat, predictions1)

disp1 = ConfusionMatrixDisplay(
    confusion_matrix=cm1,
    display_labels=labels
)

fig_cm, ax = plt.subplots(figsize=(10,10))

disp1.plot(
    ax=ax,
    cmap='Blues',
    xticks_rotation='vertical',
    colorbar=False
)

plt.title('Confusion Matrix - MODEL 2')
plt.show()

# MODEL 2
cm2 = confusion_matrix(y_test_flat, predictions2)

disp2 = ConfusionMatrixDisplay(
    confusion_matrix=cm2,
    display_labels=labels
)

fig_cm, ax = plt.subplots(figsize=(10,10))

disp2.plot(
    ax=ax,
    cmap='Greens',
    xticks_rotation='vertical',
    colorbar=False
)

plt.title('Confusion Matrix - MODEL 2')
plt.show()

In [ ]:
# ============================================================
# ACCURACY COMPARISON
# ============================================================

print("\n==============================")
print("FINAL COMPARISON")
print("==============================")

print(f"MODEL 1 ACCURACY: {accuracy1:.4f}")
print(f"MODEL 2 ACCURACY: {accuracy2:.4f}")

if accuracy2 > accuracy1:
    print("\nBest model: MODEL 2")
else:
    print("\nBest model: MODEL 1")